# 02 — Data Augmentation Tradizionale

Partendo dal dataset preprocessato (notebook 01), questo notebook:
1. importa il dataset da Drive (o lo usa localmente se già presente)
2. applica augmentation tradizionale (contrasto, luminosità, rumore) alle sole immagini **positive** del train set
3. salva le immagini augmentate in `data/real_augmented/`
4. produce `metadata.csv` con path relativi a `PROJECT_ROOT`, così che le immagini reali restino in `data/processed/` senza duplicazione

## 1. Import e configurazione path

In [ ]:
from pathlib import Path
from datetime import datetime
import shutil
import subprocess
import sys
import zipfile
import os
import json

import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

try:
    import gdown
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "gdown"])
    import gdown


PROJECT_NAME = "MammoDiffusion"

# Su Colab/Drive impostare ad esempio:
# PROJECT_ROOT_OVERRIDE = "/content/drive/MyDrive/MammoDiffusion"
PROJECT_ROOT_OVERRIDE = None


def find_project_root(project_name=PROJECT_NAME, override=PROJECT_ROOT_OVERRIDE):
    """Trova automaticamente la root del progetto."""
    if override is not None:
        root = Path(override).expanduser().resolve()
        if not root.exists():
            raise FileNotFoundError(f"PROJECT_ROOT_OVERRIDE non esiste: {root}")
        return root

    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if candidate.name == project_name:
            return candidate
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    for candidate in [
        cwd / project_name,
        Path("/content") / project_name,
        Path("/content/drive/MyDrive") / project_name,
        Path.home() / project_name,
        Path.home() / "Progetto" / project_name,
    ]:
        if candidate.exists():
            return candidate.resolve()

    raise FileNotFoundError(
        "Non riesco a trovare la root del progetto MammoDiffusion.\n"
        "Esegui il notebook dalla repo clonata oppure imposta PROJECT_ROOT_OVERRIDE."
    )


PROJECT_ROOT = find_project_root()
NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))

try:
    from eco_tracker import measure_sustainability
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q", "codecarbon", "psutil"
    ])
    from eco_tracker import measure_sustainability

DATA_DIR = PROJECT_ROOT / "data"
DATA_PROCESSED_DIR = DATA_DIR / "processed"
DATA_AUG = DATA_DIR / "real_augmented"
ARCHIVES_DIR = DATA_DIR / "archives"

RESULTS_DIR = PROJECT_ROOT / "results"
AUG_RESULTS_DIR = RESULTS_DIR / "02_data_augmentation"
AUG_PLOTS_DIR = AUG_RESULTS_DIR / "plots"
AUG_METRICS_DIR = AUG_RESULTS_DIR / "metrics"
AUG_ECOTRACKER_DIR = AUG_RESULTS_DIR / "ecotracker"
AUGMENTATION_SUMMARY_PATH = AUG_METRICS_DIR / "augmentation_summary.json"
AUGMENTATION_ECOTRACKER_PATH = AUG_ECOTRACKER_DIR / "augmentation_ecotracker.json"

for directory in [
    DATA_PROCESSED_DIR,
    DATA_AUG,
    ARCHIVES_DIR,
    AUG_PLOTS_DIR,
    AUG_METRICS_DIR,
    AUG_ECOTRACKER_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)


# DATASET PREPROCESSATO DA DRIVE
# Lo ZIP contiene: train/0/, train/1/, val/0/, val/1/, test/0/, test/1/, metadata/*.csv
PROCESSED_DRIVE_ID = "1qQral_BIBlMl0QN3PllJukdYTOmNGWr3"
PROCESSED_ZIP_PATH = ARCHIVES_DIR / "processed.zip"
PROCESSED_DIR = DATA_PROCESSED_DIR

# True = riscarica e riestrai tutto. False = usa l'esistente se presente.
FORCE_REDOWNLOAD = False

print("PROJECT_ROOT:", PROJECT_ROOT)
print("PROCESSED_DIR:", PROCESSED_DIR)
print("DATA_AUG:", DATA_AUG)

## 2. Import dataset preprocessato da Drive

In [ ]:
IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
EXPECTED_SPLITS = ["train", "val", "test"]
EXPECTED_LABELS = ["0", "1"]
REQUIRED_METADATA = ["all_processed.csv", "train.csv", "val.csv", "test.csv"]


def count_images(folder):
    """Conta le immagini dentro una cartella, includendo sottocartelle."""
    folder = Path(folder)
    if not folder.is_dir():
        return 0
    return sum(
        1 for path in folder.rglob("*")
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    )


def get_split_label_counts(processed_dir):
    """Conta immagini in <split>/<label>/."""
    processed_dir = Path(processed_dir).resolve()
    rows = []
    for split in EXPECTED_SPLITS:
        for label in EXPECTED_LABELS:
            rows.append({
                "split": split,
                "label": int(label),
                "folder": str(processed_dir / split / label),
                "n_images": count_images(processed_dir / split / label),
            })
    return pd.DataFrame(rows)


def processed_dataset_ready(processed_dir):
    """True se la struttura train/0, train/1, val/0, ... esiste con immagini."""
    processed_dir = Path(processed_dir).resolve()
    if not processed_dir.exists():
        return False
    counts_df = get_split_label_counts(processed_dir)
    ready = (counts_df["n_images"] > 0).all()
    if not ready and any((processed_dir / split).exists() for split in EXPECTED_SPLITS):
        print("Dataset preprocessato trovato, ma struttura incompleta:")
        print(counts_df[["split", "label", "n_images"]].to_string(index=False))
    return bool(ready)


def metadata_complete(processed_dir):
    """True se metadata/ contiene tutti i CSV attesi."""
    metadata_dir = Path(processed_dir) / "metadata"
    return all((metadata_dir / name).exists() for name in REQUIRED_METADATA)


def parse_filename_metadata(image_path):
    """Ricava metadata dal nome file: patient_id_image_id_laterality_view.png"""
    parts = Path(image_path).stem.split("_")
    patient_id = parts[0] if len(parts) >= 1 else Path(image_path).stem
    image_id = parts[1] if len(parts) >= 2 else Path(image_path).stem
    laterality = parts[2] if len(parts) >= 3 else "unknown"
    view = parts[3] if len(parts) >= 4 else "unknown"
    return patient_id, image_id, laterality, view


def rebuild_metadata_from_folders(processed_dir):
    """Ricostruisce i CSV metadata da train/val/test/0-1."""
    processed_dir = Path(processed_dir).resolve()
    metadata_dir = processed_dir / "metadata"
    metadata_dir.mkdir(parents=True, exist_ok=True)
    rows = []
    for split in EXPECTED_SPLITS:
        for label_name in EXPECTED_LABELS:
            label_dir = processed_dir / split / label_name
            label = int(label_name)
            for image_path in sorted(label_dir.rglob("*")):
                if not image_path.is_file() or image_path.suffix.lower() not in IMAGE_EXTENSIONS:
                    continue
                patient_id, image_id, laterality, view = parse_filename_metadata(image_path)
                rows.append({
                    "patient_id": str(patient_id),
                    "image_id": str(image_id),
                    "laterality": str(laterality),
                    "view": str(view),
                    "label": label,
                    "cancer": label,
                    "patient_label": label,
                    "split": split,
                    "source": "real",
                    "original_path": "",
                    "processed_path": str(image_path),
                })
    if not rows:
        raise FileNotFoundError(f"Nessuna immagine valida trovata in {processed_dir}")
    df = pd.DataFrame(rows).sort_values(
        ["split", "label", "patient_id", "image_id"]
    ).reset_index(drop=True)
    df.to_csv(metadata_dir / "all_processed.csv", index=False)
    for split in EXPECTED_SPLITS:
        df[df["split"] == split].reset_index(drop=True).to_csv(
            metadata_dir / f"{split}.csv", index=False,
        )
    print("Metadata CSV ricostruiti in:", metadata_dir)
    print(pd.crosstab(df["split"], df["label"]))


def ensure_metadata(processed_dir):
    if metadata_complete(processed_dir):
        print("Metadata CSV già presenti:", Path(processed_dir) / "metadata")
        return
    print("Metadata CSV mancanti: li ricostruisco da train/val/test/0-1.")
    rebuild_metadata_from_folders(processed_dir)


def download_processed_zip():
    if PROCESSED_ZIP_PATH.exists() and not FORCE_REDOWNLOAD:
        if zipfile.is_zipfile(PROCESSED_ZIP_PATH):
            print("Archivio già presente, salto download:", PROCESSED_ZIP_PATH)
            return
        print("Archivio presente ma non valido: lo riscarico.")
        PROCESSED_ZIP_PATH.unlink()
    elif PROCESSED_ZIP_PATH.exists():
        PROCESSED_ZIP_PATH.unlink()

    print("Download di processed.zip da Google Drive...")
    gdown.download(id=PROCESSED_DRIVE_ID, output=str(PROCESSED_ZIP_PATH), quiet=False)

    if not PROCESSED_ZIP_PATH.exists() or PROCESSED_ZIP_PATH.stat().st_size == 0:
        raise RuntimeError("Download fallito. Controlla la condivisione del file Drive.")
    if not zipfile.is_zipfile(PROCESSED_ZIP_PATH):
        raise RuntimeError(f"Il file scaricato non è uno ZIP valido: {PROCESSED_ZIP_PATH}")
    print("Download completato:", PROCESSED_ZIP_PATH)


def clear_processed_dir():
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    for item in PROCESSED_DIR.iterdir():
        if item.is_dir():
            shutil.rmtree(item)
        else:
            item.unlink()


def extract_processed_zip():
    print("Estrazione di processed.zip dentro:", PROCESSED_DIR)
    clear_processed_dir()
    with zipfile.ZipFile(PROCESSED_ZIP_PATH, "r") as zip_ref:
        zip_ref.extractall(PROCESSED_DIR)


def prepare_processed_dataset():
    """Usa data/processed se pronta, altrimenti scarica ed estrae da Drive."""
    if processed_dataset_ready(PROCESSED_DIR) and not FORCE_REDOWNLOAD:
        print("Dataset preprocessato già presente: salto download ed estrazione.")
        ensure_metadata(PROCESSED_DIR)
        return PROCESSED_DIR.resolve()

    PROCESSED_ZIP_PATH.parent.mkdir(parents=True, exist_ok=True)
    download_processed_zip()
    extract_processed_zip()

    if not processed_dataset_ready(PROCESSED_DIR):
        counts_df = get_split_label_counts(PROCESSED_DIR)
        print(counts_df[["split", "label", "folder", "n_images"]].to_string(index=False))
        raise FileNotFoundError(
            "processed.zip estratto, ma la struttura non è quella attesa.\n"
            "Lo ZIP deve contenere direttamente train/0, train/1, val/0, val/1, "
            "test/0, test/1 e metadata/*.csv."
        )

    ensure_metadata(PROCESSED_DIR)
    print("Dataset preprocessato pronto.")
    print("DATASET_ROOT:", PROCESSED_DIR.resolve())
    print("\nConteggio immagini:")
    print(get_split_label_counts(PROCESSED_DIR)[["split", "label", "n_images"]].to_string(index=False))
    return PROCESSED_DIR.resolve()


DATASET_ROOT = prepare_processed_dataset()

METADATA_DIR = DATASET_ROOT / "metadata"
ALL_CSV_PATH = METADATA_DIR / "all_processed.csv"
TRAIN_CSV_PATH = METADATA_DIR / "train.csv"
VAL_CSV_PATH = METADATA_DIR / "val.csv"
TEST_CSV_PATH = METADATA_DIR / "test.csv"

print("\nCSV metadata:")
print("ALL_CSV_PATH:", ALL_CSV_PATH)
print("TRAIN_CSV_PATH:", TRAIN_CSV_PATH)

## 3. Caricamento e validazione metadata

In [ ]:
def load_metadata(csv_path, dataset_root):
    df = pd.read_csv(csv_path).copy()

    required_cols = ["patient_id", "image_id", "label", "split", "processed_path"]
    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Mancano colonne obbligatorie in {csv_path}: {missing_cols}")

    df["patient_id"] = df["patient_id"].astype(str)
    df["image_id"] = df["image_id"].astype(str)
    df["label"] = df["label"].astype(int)
    df["split"] = df["split"].astype(str)

    # normalizza separatori Windows → Unix
    df["filename"] = df["processed_path"].apply(
        lambda p: str(p).replace("\\", "/").split("/")[-1]
    )
    # ricostruisce path locale corretto
    df["processed_path"] = df.apply(
        lambda row: str(
            dataset_root / row["split"] / str(row["label"]) / row["filename"]
        ),
        axis=1
    )

    df["file_exists"] = df["processed_path"].apply(lambda p: Path(p).exists())
    missing_files = df[~df["file_exists"]]

    if len(missing_files) > 0:
        print(f"Attenzione: {len(missing_files)} immagini non trovate.")
        display(missing_files[["patient_id", "image_id", "split", "label", "filename", "processed_path"]].head(10))
        raise FileNotFoundError("Alcune immagini preprocessate non sono state trovate.")

    df = df.drop(columns=["file_exists"])
    return df

In [ ]:
processed_df = load_metadata(ALL_CSV_PATH, DATASET_ROOT)
train_df = load_metadata(TRAIN_CSV_PATH, DATASET_ROOT)
val_df = load_metadata(VAL_CSV_PATH, DATASET_ROOT)
test_df = load_metadata(TEST_CSV_PATH, DATASET_ROOT)

print("Dataset caricato correttamente.")
print("Totale:", len(processed_df))
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

print("\nDistribuzione split/label:")
print(pd.crosstab(processed_df["split"], processed_df["label"]))

display(processed_df.head())

## 4. Verifica visiva campione

In [ ]:
sample_df = processed_df.sample(min(6, len(processed_df)), random_state=42)

plt.figure(figsize=(12, 8))

for i, (_, row) in enumerate(sample_df.iterrows(), start=1):
    img = Image.open(row["processed_path"]).convert("L")
    plt.subplot(2, 3, i)
    plt.imshow(img, cmap="gray")
    plt.title(f'split={row["split"]} | label={row["label"]}')
    plt.axis("off")

plt.tight_layout()
plt.show()

for _, row in sample_df.iterrows():
    img = Image.open(row["processed_path"])
    print(
        Path(row["processed_path"]).name,
        "| mode:", img.mode,
        "| size:", img.size,
        "| label:", row["label"],
        "| split:", row["split"]
    )

## 5. Augmentation tradizionale

Solo le immagini **positive** del train set vengono augmentate (contrasto, luminosità, rumore leggero).

**Le immagini reali restano in `data/processed/train/<label>/` e non vengono copiate**: il CSV le referenzia con path relativi a `PROJECT_ROOT`.
**Le immagini augmentate** vengono salvate in `data/real_augmented/`.

Nel notebook 03, leggendo il CSV, le immagini saranno caricate dalle rispettive cartelle senza duplicazione.

In [ ]:
# Quante copie augmentate per ogni immagine positiva (0 = nessuna)
POSITIVE_AUGMENT_COPIES = 3

# Se True, cancella e ricrea data/real_augmented/ (le immagini reali in
# data/processed/ NON vengono toccate)
RESET_DATASET = True

# Solo train: val/test devono restare non contaminati
source_df = train_df.copy().reset_index(drop=True)

required_cols = ["patient_id", "image_id", "label", "split", "processed_path"]
missing_cols = [col for col in required_cols if col not in source_df.columns]
if missing_cols:
    raise ValueError(f"Mancano colonne obbligatorie in train_df: {missing_cols}")

print("Distribuzione train originale:")
print(source_df["label"].value_counts())


def mild_positive_augmentation(img, aug_idx):
    """
    Augmentation leggera per mammografie positive.
    Non fa flip, perché il preprocessing ha già normalizzato il tessuto verso sinistra.
    Applica solo piccole variazioni di contrasto/luminosità/rumore.
    """
    arr = np.array(img).astype(np.float32)
    rng = np.random.default_rng(42 + aug_idx)

    contrast = rng.uniform(0.90, 1.10)
    brightness = rng.uniform(-8, 8)
    noise = rng.normal(loc=0.0, scale=2.0, size=arr.shape)

    mean = arr.mean()
    arr = (arr - mean) * contrast + mean
    arr = arr + brightness + noise
    arr = np.clip(arr, 0, 255).astype(np.uint8)

    return Image.fromarray(arr, mode="L")


with measure_sustainability(label="traditional_positive_augmentation", sample_interval=0.1) as eco_aug:
    if RESET_DATASET and DATA_AUG.exists():
        shutil.rmtree(DATA_AUG)

    DATA_AUG.mkdir(parents=True, exist_ok=True)

    metadata_rows = []
    out_idx = 0

    for row_idx, row in source_df.iterrows():
        label = int(row["label"])
        src_path = Path(row["processed_path"]).resolve()

        if not src_path.exists():
            raise FileNotFoundError(f"Immagine non trovata: {src_path}")

        # Le immagini reali NON vengono copiate: il CSV punta al file originale
        # in data/processed/train/<label>/. file_name è relativo a PROJECT_ROOT.
        real_rel_path = src_path.relative_to(PROJECT_ROOT).as_posix()
        metadata_rows.append({
            "file_name": real_rel_path,
            "label": label,
            "patient_id": str(row["patient_id"]),
            "image_id": str(row["image_id"]),
            "source": "real",
            "original_processed_path": str(src_path),
        })

        # Augmentation solo per classe positiva -> salvate in DATA_AUG.
        if label == 1 and POSITIVE_AUGMENT_COPIES > 0:
            img_l = Image.open(src_path).convert("L")
            for aug_num in range(POSITIVE_AUGMENT_COPIES):
                aug_img_l = mild_positive_augmentation(
                    img_l, aug_idx=(row_idx * 100 + aug_num)
                )

                aug_name = f"mammo_{out_idx:06d}_label1_aug{aug_num}.png"
                aug_path = DATA_AUG / aug_name
                aug_img_l.save(aug_path)

                aug_rel_path = aug_path.relative_to(PROJECT_ROOT).as_posix()
                metadata_rows.append({
                    "file_name": aug_rel_path,
                    "label": 1,
                    "patient_id": str(row["patient_id"]),
                    "image_id": str(row["image_id"]),
                    "source": "positive_augmentation",
                    "original_processed_path": str(src_path),
                })

                out_idx += 1

    augmented_df = pd.DataFrame(metadata_rows)
    metadata_csv_path = DATA_AUG / "metadata.csv"
    augmented_df.to_csv(metadata_csv_path, index=False)

label_counts = {
    str(int(label)): int(count)
    for label, count in augmented_df["label"].value_counts().sort_index().items()
}
source_counts = {
    str(source): int(count)
    for source, count in augmented_df["source"].value_counts().items()
}

eco_record = eco_aug.metrics.to_dict()
eco_record.update({
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "record_type": "traditional_positive_augmentation",
    "positive_augment_copies": POSITIVE_AUGMENT_COPIES,
    "reset_dataset": RESET_DATASET,
    "source_metadata": str(TRAIN_CSV_PATH),
    "output_metadata": str(metadata_csv_path),
    "output_dir": str(DATA_AUG),
    "n_source_train_rows": int(len(source_df)),
    "n_metadata_rows": int(len(augmented_df)),
    "n_augmented_images": int(source_counts.get("positive_augmentation", 0)),
    "label_counts": label_counts,
    "source_counts": source_counts,
    "status": "completed",
})

augmentation_summary = {
    "timestamp": datetime.now().isoformat(timespec="seconds"),
    "notebook": "02_Data_Augmentation_Trad",
    "positive_augment_copies": POSITIVE_AUGMENT_COPIES,
    "reset_dataset": RESET_DATASET,
    "source_metadata": str(TRAIN_CSV_PATH),
    "output_metadata": str(metadata_csv_path),
    "output_dir": str(DATA_AUG),
    "n_source_train_rows": int(len(source_df)),
    "n_metadata_rows": int(len(augmented_df)),
    "label_counts": label_counts,
    "source_counts": source_counts,
    "ecotracker_json": str(AUGMENTATION_ECOTRACKER_PATH),
}

with AUGMENTATION_ECOTRACKER_PATH.open("w", encoding="utf-8") as handle:
    json.dump(eco_record, handle, indent=2, ensure_ascii=False)
with AUGMENTATION_SUMMARY_PATH.open("w", encoding="utf-8") as handle:
    json.dump(augmentation_summary, handle, indent=2, ensure_ascii=False)

print("Dataset augmentato creato.")
print("Cartella augmented:", DATA_AUG)
print("metadata.csv:", metadata_csv_path)
print("Summary JSON:", AUGMENTATION_SUMMARY_PATH)
print("EcoTracker JSON:", AUGMENTATION_ECOTRACKER_PATH)
print("\nMetriche eco-tracking:", eco_aug.metrics)
print("\nDistribuzione label:")
print(augmented_df["label"].value_counts())
print("\nDistribuzione source:")
print(augmented_df["source"].value_counts())

display(augmented_df.head())

## 6. EDA — Plot per la presentazione

Plot riassuntivi della pipeline di augmentation tradizionale. Vengono salvati in `results/02_data_augmentation/plots/` e usano lo stesso stile EDA delle visualizzazioni del notebook 01.

In [ ]:
# Helper condivisi per le visualizzazioni EDA del notebook 02
PLOTS_DIR = AUG_PLOTS_DIR
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

EDA_LABEL_ORDER = [0, 1]
EDA_LABEL_NAMES = {0: "negative", 1: "positive"}
EDA_LABEL_COLORS = {0: "#4c78a8", 1: "#e45756"}
EDA_SPLIT_ORDER = ["train", "val", "test"]
EDA_SOURCE_COLORS = {"real": "#4c78a8", "positive_augmentation": "#e45756"}


def show_and_save_aug_plot(fig, filename):
    output_path = PLOTS_DIR / filename
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    print(f"Salvato: {output_path}")
    plt.show()
    plt.close(fig)


def set_annotated_bar_ylim(ax, values, top_margin=0.18):
    values_array = np.asarray(values, dtype=float).ravel()
    y_max = float(values_array.max()) if values_array.size else 0.0
    ax.set_ylim(0, y_max * (1.0 + top_margin) if y_max > 0 else 1.0)


def annotate_bars(ax, bars, fmt="{:,}", pct_of=None):
    """Annota le barre con il valore (e opzionalmente la percentuale su pct_of)."""
    for bar in bars:
        height = bar.get_height()
        text = fmt.format(int(height)).replace(",", ".")
        if pct_of:
            pct = (height / pct_of * 100) if pct_of else 0.0
            text = f"{text}\n({pct:.1f}%)"
        ax.annotate(
            text,
            xy=(bar.get_x() + bar.get_width() / 2, height),
            xytext=(0, 3),
            textcoords="offset points",
            ha="center", va="bottom", fontsize=10,
        )


print("PLOTS_DIR:", PLOTS_DIR)

In [ ]:
# EDA Aug Tappa 1 - distribuzione classi nel train: pre vs post augmentation
# Lettura diretta da source_df (pre) e augmented_df (post), nessun ricalcolo.
_train_pre = source_df["label"].value_counts().sort_index()
_train_post = augmented_df["label"].value_counts().sort_index()

pre_vals = [int(_train_pre.get(label, 0)) for label in EDA_LABEL_ORDER]
post_vals = [int(_train_post.get(label, 0)) for label in EDA_LABEL_ORDER]

x_positions = np.arange(len(EDA_LABEL_ORDER))
width = 0.36

fig, ax = plt.subplots(figsize=(9, 5))
pre_bars = ax.bar(
    x_positions - width / 2, pre_vals, width,
    label="pre augmentation", color="#9aa0a6",
)
post_bars = ax.bar(
    x_positions + width / 2, post_vals, width,
    label="post augmentation", color="#e45756",
)
set_annotated_bar_ylim(ax, pre_vals + post_vals)
annotate_bars(ax, pre_bars)
annotate_bars(ax, post_bars)

ax.set_title("Train set - distribuzione per classe prima e dopo augmentation tradizionale")
ax.set_xlabel("Classe")
ax.set_ylabel("Numero di immagini")
ax.set_xticks(x_positions)
ax.set_xticklabels([EDA_LABEL_NAMES[label] for label in EDA_LABEL_ORDER])
ax.legend(title="Fase")
ax.grid(axis="y", alpha=0.25)

show_and_save_aug_plot(fig, "train_class_balance_pre_post.png")

# Stampa anche il ratio neg:pos prima e dopo
ratio_pre = pre_vals[0] / max(pre_vals[1], 1)
ratio_post = post_vals[0] / max(post_vals[1], 1)
print(f"Ratio negative:positive pre  augmentation: {ratio_pre:.2f}:1")
print(f"Ratio negative:positive post augmentation: {ratio_post:.2f}:1")


In [ ]:
# EDA Aug Tappa 2 - composizione del dataset augmentato (real vs positive_augmentation)
# Conteggi a livello immagine sul CSV finale prodotto in DATA_AUG.
_source_counts = augmented_df["source"].value_counts()
_source_order = [s for s in ["real", "positive_augmentation"] if s in _source_counts.index]
_source_values = [int(_source_counts[s]) for s in _source_order]
_total_source = sum(_source_values)

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(
    _source_order, _source_values,
    color=[EDA_SOURCE_COLORS.get(s, "#888888") for s in _source_order],
)
set_annotated_bar_ylim(ax, _source_values)
annotate_bars(ax, bars, pct_of=_total_source)

ax.set_title("Composizione del dataset augmentato (train set)", fontsize=13, pad=18)
ax.set_xlabel("Sorgente")
ax.set_ylabel("Numero di immagini")
ax.grid(axis="y", alpha=0.25)

show_and_save_aug_plot(fig, "source_breakdown.png")


In [ ]:
# EDA Aug Tappa 3 - distribuzione train/val/test per classe sul dataset preprocessato
# Lettura da processed_df gia' validato in sezione 3, nessun ricalcolo.
_balance = (
    processed_df.groupby(["split", "label"])
    .size()
    .unstack(fill_value=0)
    .reindex(index=EDA_SPLIT_ORDER, fill_value=0)
    .reindex(columns=EDA_LABEL_ORDER, fill_value=0)
)

x_positions = np.arange(len(EDA_SPLIT_ORDER))
width = 0.36

fig, ax = plt.subplots(figsize=(9, 5))
neg_bars = ax.bar(
    x_positions - width / 2, _balance[0].to_numpy(), width,
    label="negative", color=EDA_LABEL_COLORS[0],
)
pos_bars = ax.bar(
    x_positions + width / 2, _balance[1].to_numpy(), width,
    label="positive", color=EDA_LABEL_COLORS[1],
)
set_annotated_bar_ylim(ax, _balance.to_numpy())
annotate_bars(ax, neg_bars)
annotate_bars(ax, pos_bars)

ax.set_title("Sbilanciamento di classe per split (dataset preprocessato)")
ax.set_xlabel("Split")
ax.set_ylabel("Numero di immagini")
ax.set_xticks(x_positions)
ax.set_xticklabels(EDA_SPLIT_ORDER)
ax.legend(title="Classe")
ax.grid(axis="y", alpha=0.25)

show_and_save_aug_plot(fig, "class_balance_per_split.png")


In [ ]:
# EDA Aug visiva: griglia originale + 3 augmentazioni su mammografie positive
# Sola lettura delle immagini reali; le augmentazioni vengono rigenerate al volo
# riusando mild_positive_augmentation (deterministica con seed).
_n_examples = 3

_positive_train = source_df[source_df["label"] == 1].reset_index(drop=True)
_sample_positives = _positive_train.sample(
    n=min(_n_examples, len(_positive_train)),
    random_state=7,
).reset_index(drop=True)

_n_cols = 1 + POSITIVE_AUGMENT_COPIES  # originale + N augmentazioni
_n_rows = len(_sample_positives)

fig, axes = plt.subplots(_n_rows, _n_cols, figsize=(_n_cols * 2.8, _n_rows * 2.9))
if _n_rows == 1:
    axes = axes.reshape(1, -1)

for r, (_, row) in enumerate(_sample_positives.iterrows()):
    src_path = Path(row["processed_path"]).resolve()
    img_l = Image.open(src_path).convert("L")

    axes[r, 0].imshow(img_l, cmap="gray")
    axes[r, 0].set_title(f"originale\np={row['patient_id']}", fontsize=9)
    axes[r, 0].axis("off")

    for c in range(POSITIVE_AUGMENT_COPIES):
        aug_img = mild_positive_augmentation(img_l, aug_idx=(r * 100 + c))
        axes[r, c + 1].imshow(aug_img, cmap="gray")
        axes[r, c + 1].set_title(f"aug {c}", fontsize=9)
        axes[r, c + 1].axis("off")

fig.suptitle(
    "Augmentation tradizionale su mammografie positive - contrasto + luminosita' + rumore",
    fontsize=13,
)
fig.tight_layout()

show_and_save_aug_plot(fig, "augmentation_examples.png")
